# 🏇 JRA 全レース取得 v2 (82カラム完全対応)

## 主な特徴
- **82カラム完全対応**: 過去成績（5走分）と血統情報を含む完全なデータ取得
- **欠損チェック機能**: 全カラムをチェックし、不完全なレースのみ再取得
- **メモリ効率化**: 定期的なガベージコレクションでColab環境に最適化
- **カラムずれ防止**: 厳密な82カラム定義で整合性を保証

## 使い方
1. Google Driveをマウント
2. 設定（年度、月、保存先）を変更
3. 実行ブロックを実行

In [ ]:
# Google Driveをマウントする場合のみ実行してください
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ========================================
# 設定（ここを変更してください）
# ========================================
YEAR = 2025          # 対象年度
START_MONTH = 1      # 開始月 (1-12)
END_MONTH = 12       # 終了月 (1-12)
SAVE_DIR = '/content/drive/MyDrive/dai-keiba/data/raw' # 保存先フォルダ

In [ ]:
import requestsfrom bs4 import BeautifulSoupimport pandas as pdimport ioimport refrom datetime import datetimeimport urllib.parseimport timeimport randomfrom tqdm.auto import tqdmimport gc# 82カラムの厳密な定義（データベースと完全一致）EXPECTED_COLUMNS = [    "日付","会場","レース番号","レース名","重賞","コースタイプ","距離","回り","天候","馬場状態",    "着順","枠","馬番","馬名","性齢","斤量","騎手","タイム","着差","人気","単勝オッズ","後3F",    "厩舎","馬体重(増減)","race_id","horse_id",    "past_1_date","past_1_rank","past_1_time","past_1_run_style","past_1_race_name","past_1_last_3f",    "past_1_horse_weight","past_1_jockey","past_1_condition","past_1_odds","past_1_weather",    "past_1_distance","past_1_course_type",    "past_2_date","past_2_rank","past_2_time","past_2_run_style","past_2_race_name","past_2_last_3f",    "past_2_horse_weight","past_2_jockey","past_2_condition","past_2_odds","past_2_weather",    "past_2_distance","past_2_course_type",    "past_3_date","past_3_rank","past_3_time","past_3_run_style","past_3_race_name","past_3_last_3f",    "past_3_horse_weight","past_3_jockey","past_3_condition","past_3_odds","past_3_weather",    "past_3_distance","past_3_course_type",    "past_4_date","past_4_rank","past_4_time","past_4_run_style","past_4_race_name","past_4_last_3f",    "past_4_horse_weight","past_4_jockey","past_4_condition","past_4_odds","past_4_weather",    "past_4_distance","past_4_course_type",    "past_5_date","past_5_rank","past_5_time","past_5_run_style","past_5_race_name","past_5_last_3f",    "past_5_horse_weight","past_5_jockey","past_5_condition","past_5_odds","past_5_weather",    "past_5_distance","past_5_course_type",    "father","mother","bms"]class RaceScraper:    def __init__(self):        self.headers = {            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"        }    def _get_soup(self, url, max_retries=3):        for attempt in range(max_retries):            try:                time.sleep(random.uniform(0.5, 1.0))                response = requests.get(url, headers=self.headers, timeout=15)                response.encoding = response.apparent_encoding                if response.status_code == 200:                    return BeautifulSoup(response.text, 'html.parser')                elif attempt < max_retries - 1:                    time.sleep(2 ** attempt)            except Exception as e:                if attempt < max_retries - 1:                    time.sleep(2 ** attempt)        return None    def get_past_races(self, horse_id, current_date, n_samples=5):        url = f"https://db.netkeiba.com/horse/result/{horse_id}/"        soup = self._get_soup(url)        if not soup:            return pd.DataFrame()        table = soup.select_one("table.db_h_race_results")        if not table:            tables = soup.find_all("table")            for t in tables:                if "着順" in t.text:                    table = t                    break        if not table:            return pd.DataFrame()        try:            df = pd.read_html(io.StringIO(str(table)))[0]            df = df.dropna(how='all')            df.columns = df.columns.astype(str).str.replace(r'\s+', '', regex=True)            if '日付' in df.columns:                df['date_obj'] = pd.to_datetime(df['日付'], format='%Y/%m/%d', errors='coerce')                df = df.dropna(subset=['date_obj'])                # 現在のレース日付より前のレースのみ                if current_date:                    df = df[df['date_obj'] < pd.to_datetime(current_date, format='%Y年%m月%d日', errors='coerce')]                df = df.sort_values('date_obj', ascending=False)            if n_samples:                df = df.head(n_samples)            if '通過' in df.columns:                df['run_style_val'] = df['通過'].apply(self.extract_run_style)            else:                df['run_style_val'] = ""            column_map = {                '日付': 'date', '天気': 'weather', 'レース名': 'race_name',                '着順': 'rank', '騎手': 'jockey', '馬場': 'condition',                'タイム': 'time', '上り': 'last_3f', '馬体重': 'horse_weight',                'run_style_val': 'run_style', '単勝': 'odds', 'オッズ': 'odds',                '距離': 'raw_distance'            }            df.rename(columns=column_map, inplace=True)            if 'raw_distance' in df.columns:                parsed = df['raw_distance'].apply(self._parse_dist)                df['course_type'] = parsed.apply(lambda x: x[0] if x else "")                df['distance'] = parsed.apply(lambda x: x[1] if x else "")            else:                df['course_type'] = ""                df['distance'] = ""            # データクリーニング            for col in ['rank', 'time', 'run_style', 'race_name', 'last_3f', 'horse_weight',                        'jockey', 'condition', 'odds', 'weather', 'distance', 'course_type', 'date']:                if col in df.columns:                    df[col] = df[col].fillna("").astype(str).str.replace('nan', '').str.strip()            return df        except Exception as e:            return pd.DataFrame()    def _parse_dist(self, x):        if not isinstance(x, str):            return ("", "")        surf = ""        dist = ""        if '芝' in x: surf = '芝'        elif 'ダ' in x: surf = 'ダート'        elif '障' in x: surf = '障害'        match = re.search(r'(\d+)', x)        if match:            dist = match.group(1)        return (surf, dist)    def extract_run_style(self, passing_str):        if not isinstance(passing_str, str):            return ""        try:            cleaned = re.sub(r'[^0-9-]', '', passing_str)            parts = [int(p) for p in cleaned.split('-') if p]            if not parts:                return ""            first_corner = parts[0]            if first_corner == 1: return "1"            elif first_corner <= 4: return "2"            elif first_corner <= 9: return "3"            else: return "4"        except:            return ""    def get_horse_profile(self, horse_id):        url = f"https://db.netkeiba.com/horse/ped/{horse_id}/"        soup = self._get_soup(url)        if not soup:            return {"father": "", "mother": "", "bms": ""}        data = {"father": "", "mother": "", "bms": ""}        try:            table = soup.select_one("table.blood_table")            if table:                rows = table.find_all("tr")                if len(rows) >= 17:                    r0 = rows[0].find_all("td")                    if r0:                        data["father"] = r0[0].text.strip().split('\n')[0].strip()                    r16 = rows[16].find_all("td")                    if len(r16) >= 2:                        data["mother"] = r16[0].text.strip().split('\n')[0].strip()                        data["bms"] = r16[1].text.strip().split('\n')[0].strip()        except:            pass        return datadef scrape_jra_race(url, existing_race_ids=None, max_retries=3):    """    Scrapes a single race page from JRA website with full 82-column support.    Returns a pandas DataFrame with past performance and pedigree data.    """    print(f"Accessing URL: {url}...")    headers = {        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"    }    scraper = RaceScraper()    for attempt in range(max_retries):        try:            response = requests.get(url, headers=headers, timeout=15)            response.encoding = 'EUC-JP'            if response.status_code != 200:                if attempt < max_retries - 1:                    wait_time = 2 ** attempt                    print(f"Status {response.status_code}, retrying in {wait_time}s...")                    time.sleep(wait_time)                    continue                else:                    print(f"Error: Status code {response.status_code}")                    return None            soup = BeautifulSoup(response.text, 'html.parser')            # --- Metadata Extraction ---            h1_elem = soup.select_one("div.header_line h1 .txt")            full_text = h1_elem.text.strip() if h1_elem else ""            if not full_text and soup.h1:                full_text = soup.h1.text.strip()            date_text = ""            venue_text = ""            race_num_text = ""            kai = "01"            day = "01"            match_date = re.search(r'(\d{4}年\d{1,2}月\d{1,2}日)', full_text)            if match_date:                date_text = match_date.group(1)            venues_str = "札幌|函館|福島|新潟|東京|中山|中京|京都|阪神|小倉"            match_meta = re.search(rf'(\d+)回({venues_str})(\d+)日', full_text)            if match_meta:                kai = f"{int(match_meta.group(1)):02}"                venue_text = match_meta.group(2)                day = f"{int(match_meta.group(3)):02}"            match_race = re.search(r'(\d+)レース', full_text)            if match_race:                r_val = int(match_race.group(1))                race_num_text = f"{r_val}R"                r_num = f"{r_val:02}"            else:                race_num_text = "10R"                r_num = "10"            race_name_text = ""            name_elem = soup.select_one(".race_name")            if name_elem:                race_name_text = name_elem.text.strip()            grade_text = ""            if "G1" in str(soup) or "ＧⅠ" in str(soup): grade_text = "G1"            elif "G2" in str(soup) or "ＧⅡ" in str(soup): grade_text = "G2"            elif "G3" in str(soup) or "ＧⅢ" in str(soup): grade_text = "G3"            header_text = soup.select_one("div.header_line").text if soup.select_one("div.header_line") else soup.text            dist_type_match = re.search(r'(芝|ダ|ダート|障害)[^0-9]*(\d+)', header_text)            course_type = ""            distance = ""            if dist_type_match:                c_val = dist_type_match.group(1)                d_val = dist_type_match.group(2)                if "芝" in c_val: course_type = "芝"                elif "ダ" in c_val: course_type = "ダート"                elif "障" in c_val: course_type = "障害"                distance = d_val            rotation = ""            rot_match = re.search(r'[（\(](右|左|直線)[）\)]', header_text)            if rot_match:                rotation = rot_match.group(1)            else:                 if "左" in header_text: rotation = "左"                 elif "右" in header_text: rotation = "右"                 elif "直線" in header_text: rotation = "直線"            weather = ""            w_match = re.search(r'天候\s*[:：]\s*(\S+)', soup.text)            if w_match:                weather = w_match.group(1).strip()            condition = ""            if course_type == "芝":                 c_match = re.search(r'芝\s*[:：]\s*(\S+)', soup.text)                 if c_match: condition = c_match.group(1).strip()            elif course_type == "ダート":                 c_match = re.search(r'ダート\s*[:：]\s*(\S+)', soup.text)                 if c_match: condition = c_match.group(1).strip()            if not condition:                 c_match_gen = re.search(r'(?:芝|ダート)\s*[:：]\s*(\S+)', soup.text)                 if c_match_gen: condition = c_match_gen.group(1).strip()            # --- Table Extraction ---            tables = soup.find_all('table')            target_table = None            for tbl in tables:                if "着順" in tbl.text and "馬名" in tbl.text:                    target_table = tbl                    break            if not target_table:                print(f"Warning: Result table not found in {url}")                return None            rows = target_table.find_all('tr')            data = []            for row in rows:                if row.find('th'):                    continue                cells = row.find_all('td')                if not cells:                    continue                def get_text(idx):                    if idx < len(cells):                        return cells[idx].get_text(strip=True)                    return ""                waku_text = ""                if len(cells) > 1:                    img = cells[1].find('img')                    if img and 'alt' in img.attrs:                        alt = img['alt']                        m = re.search(r'枠(\d+)', alt)                        if m:                            waku_text = m.group(1)                        else:                            waku_text = alt                horse_id = ""                if len(cells) > 3:                    a_tag = cells[3].find('a')                    if a_tag and 'href' in a_tag.attrs:                        href = a_tag['href']                        m = re.search(r'/horse/(\d+)', href)                        if m:                            horse_id = m.group(1)                row_data = {                    '着順': get_text(0),                    '枠': waku_text,                    '馬番': get_text(2),                    '馬名': get_text(3),                    'horse_id': horse_id,                    '性齢': get_text(4),                    '斤量': get_text(5),                    '騎手': get_text(6),                    'タイム': get_text(7),                    '着差': get_text(8),                    '後3F': get_text(10),                    '厩舎': get_text(12),                    '馬体重(増減)': get_text(11),                    '人気': get_text(13),                    '単勝オッズ': "0.0"                }                data.append(row_data)            df = pd.DataFrame(data)            # Add Metadata            df['日付'] = date_text            df['会場'] = venue_text            df['レース番号'] = race_num_text            df['レース名'] = race_name_text            df['重賞'] = grade_text            df['距離'] = distance            df['コースタイプ'] = course_type            df['天候'] = weather            df['馬場状態'] = condition            df['回り'] = rotation            # ID Generation            place_map = {                "札幌": "01", "函館": "02", "福島": "03", "新潟": "04", "東京": "05",                "中山": "06", "中京": "07", "京都": "08", "阪神": "09", "小倉": "10"            }            p_code = place_map.get(venue_text, "00")            year = "2025"            if date_text:                year = date_text[:4]            generated_id = f"{year}{p_code}{kai}{day}{r_num}"            # SKIP CHECK            if existing_race_ids and generated_id in existing_race_ids:                print(f"Skipping {generated_id} (Already exists)")                return None            df['race_id'] = generated_id            # Cleanups            if '単勝オッズ' in df.columns:                df['単勝オッズ'] = pd.to_numeric(df['単勝オッズ'], errors='coerce').fillna(0.0).astype(str)            # === 過去成績と血統の取得 ===            print(f"  Fetching past performance & pedigree for {len(df)} horses...")            for idx in range(len(df)):                horse_id = df.at[idx, 'horse_id']                # 過去成績の初期化                for n in range(1, 6):                    prefix = f"past_{n}_"                    for field in ['date', 'rank', 'time', 'run_style', 'race_name', 'last_3f',                                  'horse_weight', 'jockey', 'condition', 'odds', 'weather',                                  'distance', 'course_type']:                        df.at[idx, prefix + field] = ""                # 血統の初期化                df.at[idx, 'father'] = ""                df.at[idx, 'mother'] = ""                df.at[idx, 'bms'] = ""                if not horse_id or not str(horse_id).isdigit():                    continue                # 過去成績取得                try:                    past_df = scraper.get_past_races(horse_id, date_text, n_samples=5)                    if not past_df.empty:                        for n, (_, past_row) in enumerate(past_df.iterrows()):                            if n >= 5:                                break                            prefix = f"past_{n+1}_"                            df.at[idx, prefix + 'date'] = past_row.get('date', "")                            df.at[idx, prefix + 'rank'] = past_row.get('rank', "")                            df.at[idx, prefix + 'time'] = past_row.get('time', "")                            df.at[idx, prefix + 'run_style'] = past_row.get('run_style', "")                            df.at[idx, prefix + 'race_name'] = past_row.get('race_name', "")                            df.at[idx, prefix + 'last_3f'] = past_row.get('last_3f', "")                            df.at[idx, prefix + 'horse_weight'] = past_row.get('horse_weight', "")                            df.at[idx, prefix + 'jockey'] = past_row.get('jockey', "")                            df.at[idx, prefix + 'condition'] = past_row.get('condition', "")                            df.at[idx, prefix + 'odds'] = past_row.get('odds', "")                            df.at[idx, prefix + 'weather'] = past_row.get('weather', "")                            df.at[idx, prefix + 'distance'] = past_row.get('distance', "")                            df.at[idx, prefix + 'course_type'] = past_row.get('course_type', "")                except Exception as e:                    pass                # 血統取得                try:                    profile = scraper.get_horse_profile(horse_id)                    if profile:                        df.at[idx, 'father'] = profile.get('father', "")                        df.at[idx, 'mother'] = profile.get('mother', "")                        df.at[idx, 'bms'] = profile.get('bms', "")                except Exception as e:                    pass            # メモリ解放            gc.collect()            # 最終カラム整合性チェック - 82カラムを厳密に保証            df = df.reindex(columns=EXPECTED_COLUMNS, fill_value="")            print(f"✅ Scraped {len(df)} rows with full 82-column data.")            return df        except Exception as e:            if attempt < max_retries - 1:                wait_time = 2 ** attempt                print(f"Error: {e}, retrying in {wait_time}s...")                time.sleep(wait_time)            else:                print(f"❌ Failed after {max_retries} attempts: {e}")                return None    return None# Parameter Map for Monthly ResultsJRA_MONTH_PARAMS = {    "2026": { "01": "E4", "02": "B2", "03": "80", "04": "4E", "05": "1C", "06": "EA", "07": "B8", "08": "86", "09": "54", "10": "22", "11": "F0", "12": "BE" },    "2025": { "01": "3F", "02": "0D", "03": "DB", "04": "A9", "05": "77", "06": "45", "07": "13", "08": "E1", "09": "AF", "10": "1E", "11": "EC", "12": "D3" },    "2024": { "01": "B3", "02": "81", "03": "4F", "04": "1D", "05": "EB", "06": "B9", "07": "87", "08": "55", "09": "23", "10": "92", "11": "60", "12": "2E" },    "2023": { "01": "27", "02": "F5", "03": "C3", "04": "91", "05": "5F", "06": "2D", "07": "FB", "08": "C9", "09": "97", "10": "06", "11": "D4", "12": "A2" },    "2022": { "01": "9B", "02": "69", "03": "37", "04": "05", "05": "D3", "06": "A1", "07": "6F", "08": "3D", "09": "0B", "10": "7A", "11": "48", "12": "16" },    "2021": { "01": "0F", "02": "DD", "03": "AB", "04": "79", "05": "47", "06": "15", "07": "E3", "08": "B1", "09": "7F", "10": "EE", "11": "BC", "12": "8A" },    "2020": { "01": "83", "02": "51", "03": "1F", "04": "ED", "05": "BB", "06": "89", "07": "57", "08": "25", "09": "F3", "10": "62", "11": "30", "12": "FE" }}def scrape_jra_year(year_str, start_date=None, end_date=None, save_callback=None, existing_race_ids=None):    """    Scrapes races for a given year and date range with progress tracking and memory efficiency.    """    if year_str not in JRA_MONTH_PARAMS:        print(f"Year {year_str} not supported in parameter map.")        return    params = JRA_MONTH_PARAMS[year_str]    base_url = "https://www.jra.go.jp/JRADB/accessS.html"    start_m = 1    end_m = 12    if start_date:        start_m = start_date.month    if end_date:        end_m = end_date.month    from datetime import date    today = date.today()    if end_date:        actual_end_date = min(end_date, today)    else:        actual_end_date = today    print(f"=== Starting JRA Bulk Scraping for {year_str} ===")    print(f"Period: {start_date or 'Start'} - {actual_end_date}")    print(f"Using random delays (1.0-2.0s) to avoid rate limiting")    if int(year_str) == today.year:        end_m = min(end_m, today.month)    elif int(year_str) > today.year:        print(f"Year {year_str} is in the future. Stopping.")        return    failed_races = []    total_processed = 0    for m in range(start_m, end_m + 1):        month = f"{m:02}"        if month not in params:            continue        suffix = params[month]        try:            ym = int(year_str + month)            prefix = "pw01skl00" if ym >= 202512 else "pw01skl10"        except:            prefix = "pw01skl10"        cname = f"{prefix}{year_str}{month}/{suffix}"        print(f"\n📅 Fetching {year_str}/{month}...")        try:            headers = {                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"            }            response = requests.post(base_url, data={"cname": cname}, headers=headers, timeout=15)            response.encoding = 'cp932'            if response.status_code != 200:                print(f"❌ Failed to fetch {cname} (Status {response.status_code})")                continue            soup = BeautifulSoup(response.text, 'html.parser')            race_cnames = []            links = soup.find_all('a')            for link in links:                onclick = link.get('onclick', '')                match = re.search(r"doAction\('[^']+',\s*'([^']+)'\)", onclick)                if match:                    c = match.group(1)                    if c.startswith('pw01srl'):                        race_cnames.append(c)            race_cnames = sorted(list(set(race_cnames)))            print(f"  Found {len(race_cnames)} race days")            for day_cname in tqdm(race_cnames, desc=f"  {year_str}/{month}", leave=False):                resp_day = requests.post(base_url, data={"cname": day_cname}, headers=headers, timeout=15)                resp_day.encoding = 'cp932'                soup_day = BeautifulSoup(resp_day.text, 'html.parser')                d_h1 = soup_day.select_one("div.header_line h1 .txt")                full_d_text = d_h1.text.strip() if d_h1 else (soup_day.h1.text.strip() if soup_day.h1 else "")                current_day_date = None                kai_str = "01"                day_str = "01"                venue_str = ""                p_code = "00"                match_day_date = re.search(r'(\d{4})年(\d{1,2})月(\d{1,2})日', full_d_text)                if match_day_date:                    y, mo, d_day = map(int, match_day_date.groups())                    current_day_date = datetime(y, mo, d_day).date()                    if start_date and current_day_date < start_date:                        continue                    if end_date and current_day_date > end_date:                        continue                venues_ptn = "札幌|函館|福島|新潟|東京|中山|中京|京都|阪神|小倉"                match_meta = re.search(rf'(\d+)回({venues_ptn})(\d+)日', full_d_text)                if match_meta:                    kai_str = f"{int(match_meta.group(1)):02}"                    venue_str = match_meta.group(2)                    day_str = f"{int(match_meta.group(3)):02}"                    place_map = {                        "札幌": "01", "函館": "02", "福島": "03", "新潟": "04", "東京": "05",                        "中山": "06", "中京": "07", "京都": "08", "阪神": "09", "小倉": "10"                    }                    p_code = place_map.get(venue_str, "00")                race_list_items = []                all_anchors = soup_day.find_all('a')                for a in all_anchors:                    onclick = a.get('onclick', '')                    match_sde = re.search(r"doAction\s*\(\s*['\"][^'\"]+['\"]\s*,\s*['\"](pw01sde[^'\"]+)['\"]\s*\)", onclick)                    href = a.get('href', '')                    final_url = ""                    if match_sde:                        final_url = f"{base_url}?CNAME={match_sde.group(1)}"                    elif 'pw01sde' in href:                        final_url = urllib.parse.urljoin(base_url, href)                    if final_url:                        txt = a.text.strip()                        img = a.find('img')                        if not txt and img and 'alt' in img.attrs:                            txt = img['alt']                        r_num = -1                        r_num_match = re.search(r'(\d+)R', txt)                        if r_num_match:                             r_num = int(r_num_match.group(1))                        race_list_items.append((final_url, r_num))                seen_urls = set()                unique_races = []                for url, r_num in race_list_items:                    if url not in seen_urls:                        unique_races.append((url, r_num))                        seen_urls.add(url)                unique_races.sort(key=lambda x: x[1])                for r_link, r_num in unique_races:                    # Pre-check skip                    if r_num != -1 and p_code != "00" and current_day_date:                         y_str = str(y)                         r_num_str = f"{r_num:02}"                         generated_id = f"{y_str}{p_code}{kai_str}{day_str}{r_num_str}"                         if existing_race_ids and generated_id in existing_race_ids:                             continue                    # Fetch with retry                    df = scrape_jra_race(r_link, existing_race_ids=existing_race_ids)                    if df is not None and not df.empty:                        if save_callback:                            save_callback(df)                        total_processed += 1                    else:                        race_id = generated_id if r_num != -1 else r_link                        failed_races.append(race_id)                    # Rate limiting with random delay                    time.sleep(random.uniform(1.0, 2.0))                    # Session keepalive & memory cleanup                    if total_processed % 5 == 0 and total_processed > 0:                        print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ {total_processed}件処理完了")                        gc.collect()  # メモリクリーンアップ        except Exception as e:            print(f"❌ Error processing month {month}: {e}")            gc.collect()    # Final summary    print(f"\n{'='*50}")    print(f"✅ スクレイピング完了")    print(f"総処理件数: {total_processed}件")    print(f"失敗件数: {len(failed_races)}件")    if failed_races:        print(f"\n⚠️ 失敗したレース:")        for race_id in failed_races[:10]:            print(f"  - {race_id}")        if len(failed_races) > 10:            print(f"  ... 他 {len(failed_races) - 10}件")

In [ ]:
# 実行ブロック（欠損チェック機能付き）import osimport pandas as pdfrom datetime import dateimport calendarif YEAR:    os.makedirs(SAVE_DIR, exist_ok=True)    save_path = os.path.join(SAVE_DIR, 'database.csv')    # 安全な追記関数（82カラム厳密保証）    def safe_append_csv(df_chunk, path):        import pandas as pd        import os        if not os.path.exists(path):            # 新規作成 - 82カラムヘッダーを確実に書き込み            df_chunk = df_chunk.reindex(columns=EXPECTED_COLUMNS, fill_value="")            df_chunk.to_csv(path, index=False)        else:            try:                # 既存ヘッダー読み込み                existing_cols = pd.read_csv(path, nrows=0).columns.tolist()                                # カラム数チェック                if len(existing_cols) != 82:                    print(f"⚠️ 警告: 既存ファイルのカラム数が{len(existing_cols)}です（期待値: 82）")                    print(f"  ファイルを確認してください: {path}")                    return                                # カラム順序を既存ファイルに合わせる                df_aligned = df_chunk.reindex(columns=existing_cols, fill_value="")                                # 追記                df_aligned.to_csv(path, mode='a', header=False, index=False)            except Exception as e:                print(f"❌ 保存エラー: {e}")    # 既存データの読み込みと欠損チェック    existing_race_ids = set()    if os.path.exists(save_path):        print('既存データを読み込み中...')        try:            existing_df = pd.read_csv(save_path, low_memory=False)            if 'race_id' in existing_df.columns:                # race_idを文字列に変換                existing_df['race_id'] = existing_df['race_id'].astype(str).str.replace(r'\.0$', '', regex=True)                # 全82カラムのチェック（欠損がない行のみ「完全」）                complete_mask = pd.Series(True, index=existing_df.index)                                # 基本情報カラムチェック                basic_cols = ['日付', '会場', 'レース番号', 'レース名', 'コースタイプ', '距離',                               '天候', '馬場状態', '馬名', 'horse_id']                for col in basic_cols:                    if col in existing_df.columns:                        complete_mask = complete_mask & existing_df[col].notna() & \                                        (existing_df[col] != '') & (existing_df[col] != 'nan')                                # 過去成績カラムチェック（past_1のみチェック - あれば全体OKとみなす）                if 'past_1_date' in existing_df.columns:                    has_past_data = existing_df['past_1_date'].notna() & \                                    (existing_df['past_1_date'] != '') & \                                    (existing_df['past_1_date'] != 'nan')                    complete_mask = complete_mask & has_past_data                                # 血統カラムチェック                if 'father' in existing_df.columns:                    has_pedigree = existing_df['father'].notna() & \                                   (existing_df['father'] != '') & \                                   (existing_df['father'] != 'nan')                    complete_mask = complete_mask & has_pedigree                complete_races = existing_df[complete_mask]['race_id'].unique()                existing_race_ids = set(complete_races)                total_races = existing_df['race_id'].nunique()                complete_count = len(existing_race_ids)                incomplete_count = total_races - complete_count                                print(f'既存データ: {total_races}レース')                print(f'  完全データ: {complete_count}レース')                print(f'  不完全データ: {incomplete_count}レース（再取得対象）')                                # カラム数確認                actual_cols = len(existing_df.columns)                if actual_cols != 82:                    print(f'⚠️ 警告: 既存データのカラム数が{actual_cols}です（期待値: 82）')                            except Exception as e:            print(f'既存データの読み込みエラー（新規作成します）: {e}')    s_date = date(int(YEAR), int(START_MONTH), 1)    last_day = calendar.monthrange(int(YEAR), int(END_MONTH))[1]    e_date = date(int(YEAR), int(END_MONTH), last_day)    today = date.today()    if e_date > today:        e_date = today    print(f'\n{YEAR}年のデータを {s_date} から {e_date} まで取得します...')    print(f'保存先: {save_path}')    print(f'完全な82カラムデータ（過去成績・血統含む）を取得します')        scrape_jra_year(        str(YEAR),        start_date=s_date,        end_date=e_date,        existing_race_ids=existing_race_ids,        save_callback=lambda df: safe_append_csv(df, save_path)    )        # 最終確認    if os.path.exists(save_path):        final_df = pd.read_csv(save_path, nrows=0)        print(f'\n✅ 完了しました。')        print(f'最終カラム数: {len(final_df.columns)}')        if len(final_df.columns) == 82:            print('🎉 カラム数が正しいです（82カラム）')        else:            print(f'⚠️ カラム数が不正です（期待: 82, 実際: {len(final_df.columns)}）')else:    print('年度が設定されていません。')